# 01 · Schema and data dictionary (Team 10, Saideep)

**Deliverable 3, part 1.** Load the team's fare file through the shared `load_data()`,
check that every column has the type and meaning we think it has, run the consistency
checks the column definitions imply, and write `results/data_dictionary.md`.

The dictionary holds **metadata and counts only** — no row values — so it is safe to commit.
Re-run this notebook (Kernel → Restart & Run All) whenever the data file changes.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

HERE = next(p for p in [Path.cwd(), Path.cwd() / "team-10-modeling" / "notebooks"]
            if (p / "load_data.py").exists())
sys.path.insert(0, str(HERE))
RESULTS = HERE.parent / "results"
RESULTS.mkdir(exist_ok=True)

from load_data import DEFAULT_FILE, TARGET, load_data

df = load_data()
raw = load_data(raw=True)
print(f"{DEFAULT_FILE}: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.dtypes

## Column meanings

The compact file is a 13-column cut of the synthetic AA fare sample. Meanings come from
`data/raw/DATA_SYNTHETIC.md`, which documents the same columns in the full file.
Grain: **one fare quote** (a priced itinerary at the moment it was searched).

In [ ]:
MEANING = {
    "days_to_departure": "Days between the quote and the departure (advance-purchase window).",
    "origin": "Origin airport (AA hubs over-represented).",
    "destination": "Destination airport.",
    "distance_miles": "Great-circle miles for the origin-destination pair.",
    "cabin": "Basic Economy / Main Cabin / Premium Economy / Business/First (ordered).",
    "departure_hour": "Scheduled departure hour, local, 24h.",
    "is_weekend_departure": "1 if the departure is Fri-Sun.",
    "is_holiday_period": "1 if the departure falls in the Thanksgiving / late-Dec / July-4 window.",
    "month": "Departure month (1-12).",
    "nonstop": "1 if the itinerary is nonstop.",
    "load_factor": "Fraction of seats already sold when the fare was quoted (0-1).",
    "competitor_min_price": "Lowest competing quote for the itinerary, USD.",
    "total_price": "**TARGET.** AA quoted total price, USD.",
}
assert set(MEANING) == set(df.columns), set(df.columns) ^ set(MEANING)

rows = []
for col in df.columns:
    rows.append({
        "column": col,
        "dtype": str(df[col].dtype),
        "role": "target" if col == TARGET else "feature",
        "n_unique": int(df[col].nunique()),
        "n_missing": int(df[col].isna().sum()),
        "pct_missing": round(100 * df[col].isna().mean(), 2),
        "meaning": MEANING[col],
    })
dictionary = pd.DataFrame(rows)
dictionary

## Consistency checks

Each rule below follows from a column's definition. A violation is either a data error or a
sign that we misunderstood the column.

In [ ]:
route_dist = df.groupby(["origin", "destination"], observed=True)["distance_miles"].nunique()
pair = df[["origin", "destination"]].astype(str)
checks = {
    "days_to_departure >= 0": (df["days_to_departure"] < 0).sum(),
    "month in 1..12": (~df["month"].between(1, 12)).sum(),
    "departure_hour in 0..23": (~df["departure_hour"].between(0, 23)).sum(),
    "0/1 flags only 0 or 1": sum((~df[c].isin([0, 1])).sum()
                                 for c in ["is_weekend_departure", "is_holiday_period", "nonstop"]),
    "load_factor within [0, 1]": (~df["load_factor"].between(0, 1)).sum(),
    "distance_miles constant per route": int((route_dist > 1).sum()),
    "origin != destination": (pair["origin"] == pair["destination"]).sum(),
    "total_price > 0": (df[TARGET] <= 0).sum(),
    "competitor_min_price > 0": (df["competitor_min_price"] <= 0).sum(),
    "total_price within 3x of competitor": ((df[TARGET] / df["competitor_min_price"] > 3)
                                            | (df[TARGET] / df["competitor_min_price"] < 1 / 3)).sum(),
    "no exact duplicate rows": df.duplicated().sum(),
    "cabin label recognised": df["cabin"].isna().sum() - (raw["cabin"] == "").sum(),
}
check_table = pd.DataFrame(
    [{"rule": k, "violating_rows": int(v), "pct": round(100 * int(v) / len(df), 2)}
     for k, v in checks.items()]
)
check_table

## What is *not* in this file

Compared with the full synthetic sample, the compact file has **no dates** (`search_date`,
`departure_date`), no `day_of_week`, no `fare_booking_class`, no seat counts, and no
`base_fare` / `taxes_fees`. Two consequences:

- The `base_fare + taxes_fees = total_price` leakage trap is already gone.
- There is **no time axis**, so a chronological train/test split, ARIMA, or any multi-year
  forecast cannot be evaluated on this file. Validation has to be random (cross-validated).

In [ ]:
def to_md(frame: pd.DataFrame) -> str:
    cols = list(frame.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for _, r in frame.iterrows():
        lines.append("| " + " | ".join(str(r[c]) for c in cols) + " |")
    return "\n".join(lines)

n_missing_cols = dictionary.loc[dictionary["n_missing"] > 0, "column"].tolist()
md = f"""# Data dictionary — Team 10

_Generated by `team-10-modeling/notebooks/01_schema_saideep.ipynb` from `data/raw/{DEFAULT_FILE}`
(loaded through `load_data()`). Do not hand-edit; re-run the notebook._

> Metadata and counts only: no row values or samples. Keep it that way.

## Overview

- **File:** `{DEFAULT_FILE}` (synthetic, not real AA data)
- **Rows:** {len(df):,}
- **Columns:** {df.shape[1]}
- **Grain:** one fare quote (a priced itinerary at the moment it was searched)
- **Target:** `{TARGET}`
- **Exact duplicate rows:** {int(df.duplicated().sum())}
- **Columns with missing values:** {", ".join(n_missing_cols) if n_missing_cols else "none"}
- **Time columns:** none (no search or departure date), so no chronological split is possible

## Columns

| column | dtype | role | n_unique | n_missing | pct_missing | meaning |
|---|---|---|---|---|---|---|
""" + "\n".join(
    f"| {r.column} | {r.dtype} | {r.role} | {r.n_unique} | {r.n_missing} | {r.pct_missing} | {r.meaning} |"
    for r in dictionary.itertuples()
) + f"""

## Consistency checks

{to_md(check_table)}

## Open questions for the PM

| column(s) | question |
|---|---|
| load_factor | Measured when the fare was quoted, or at departure? If at departure it is leakage. |
| competitor_min_price | Captured at the same moment, for the same cabin, as the AA quote? Is it available to us when we price? |
| total_price | Per passenger, one-way, USD? The lowest available fare or the fare actually sold? |
| (dates) | Can we get `search_date` / `departure_date` back? Without them nothing time-based can be validated. |
"""
(RESULTS / "data_dictionary.md").write_text(md, encoding="utf-8")
print("wrote", RESULTS / "data_dictionary.md")